### Import Dependecies

In [1]:
import os

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig
from google.genai import types

from utils.utils import get_tool_descriptions, format_ai_message
from utils.tools import check_warehouse_availability, reserve_warehouse_items

### ADK Agent

In [2]:
model = LiteLlm(
    model="openai/gpt-5.4-mini",
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    model=model,
    tools=[check_warehouse_availability, reserve_warehouse_items],
    description="An agent that can check the availability of items in the warehouses and reserve them.",
    instruction="""
You are a part of the shopping assistant that can manage available inventory in the warehouses.

You will be given a conversation history and a list of tools, your task is to perform actions requested by the latest user query. Answer part of the query that you can answer with the available tools.

Instructions:
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combine the required quantity from the different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.
"""
)

### ADK Session

In [5]:
session_service =  InMemorySessionService()

In [6]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="session_1"
)

Session(id='session_1', app_name='warehouse_app', user_id='user_1', state={}, events=[], last_update_time=1779199889.9019659)

In [8]:
runner = Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app",
)

In [10]:
message = types.Content(
    role="user",
    parts=[types.Part(text="How many of B0BP68QCZ2 available in all warehouses?")]
)

In [11]:
result = runner.run(
    user_id="user_1",
    session_id="session_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=3
    )
)

In [13]:
for event in result:
    print("==============================")
    print(event)

19:28:35 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
19:28:36 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


model_version='gpt-5.4-mini-2026-03-17' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_4GUVbzK7OZTTU6amkTToZANG',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=36,
  prompt_token_count=604,
  total_token_count=640
) live_session_resumption_update=None live_session_id=None go_away=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None invocation_id='e-9fc9456d-1d5d-4230-8ea1-c38d0cd9d805' author='warehouse_manager_agent' actions=Ev

In [21]:
session_service =  InMemorySessionService()

async def ask_warehouse_agent(query: str, session_id: str):

    # Check if session exist, only create if it doesn't
    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id="user_1",
            session_id=session_id
        )

    runner = Runner(
        agent=warehouse_agent,
        app_name="warehouse_app",
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    final_text = ""

    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=3
        )
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_text += part.text
            break
    
    return final_text

In [22]:
answer = await ask_warehouse_agent("What is the availability of B0BP68QCZ2 in all of the warehouses?", session_id = "123")

In [23]:
print(answer)

B0BP68QCZ2 is available in all warehouses, and each warehouse can fulfill at least 1 unit.

Availability by warehouse:
- DE-BER-01 — Berlin Distribution Center: 15
- FR-LY0-01 — Lyon Regional Warehouse: 96
- DE-MUN-01 — Munich Logistics Hub: 42
- FR-PAR-01 — Paris Central Depot: 25
- FR-MAR-01 — Marseille Mediterranean Hub: 4
- DE-HAM-01 — Hamburg North Warehouse: 65

If you want, I can also help reserve a specific quantity from the closest warehouse.


In [24]:
answer = await ask_warehouse_agent("Can you reserve 10 items from Lyon?", session_id = "123")

In [25]:
print(answer)

Yes — 10 units of B0BP68QCZ2 have been reserved from Lyon Regional Warehouse (FR-LY0-01).

Reserved:
- Product: B0BP68QCZ2
- Quantity: 10
- Warehouse: Lyon Regional Warehouse, Lyon, France
